# Common Libraries

In [1]:
import os, sys, shutil, functools
import numpy as np

if shutil.which("nvidia-smi") is not None:
    os.environ["MUJOCO_GL"] = "egl"

# Mujoco
import mujoco

# GPU setting
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# Jax
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".60" 

import jax
jax.config.update("jax_enable_x64", True)
from jax import numpy as jnp
from jaxtyping import Integer, Float, Array

# Mujoco jax
from mujoco import mjx
from mujoco.mjx import PyTreeNode

# Custom Libraries

In [2]:
sys.path.append("/home/seojin/Seojin_commonTool/Module")
from vis_3D import Plotter3D
from sj_datastructure import make_3d_dataset

In [16]:
def reorder_sites(mjx_data: mjx.Data, site_reorder: Integer[Array, "sites"] | None) -> mjx.Data:
    if site_reorder is None:
        return mjx_data
    else:
        site_xpos = mjx_data.site_xpos[site_reorder]
        return mjx_data.replace(site_xpos=site_xpos)

class State(PyTreeNode):
    """
    State variables needed for forward kinematics

    This is a subset of the attributes available in `mjx.Data`, but including
    all of them in the full output dramatically slows things down, even if they
    are not used.

    Attributes:
        qpos: joint position                                          (nq,)
        qvel: joint velocity                                          (nv,)
        xpos:  Cartesian position of body frame                       (nbody, 3)
        site_xpos: Cartesian site position                            (nsite, 3)

    """

    qpos: jax.Array
    qvel: jax.Array
    xpos: jax.Array
    site_xpos: jax.Array
    efc_force: jax.Array | None
    
@functools.partial(jax.jit, static_argnames=("return_contact_forces"))
def reduce_state(mjx_data: mjx.Data, return_contact_forces: bool = False) -> State:
    """
    Reduce the mjx data to a state object

    Empirically, this seems to improve performance over returning the full mjx data
    and then only using a subset.

    Args:
        mjx_data: The mujoco data
        return_contact_forces: Return the contact forces or not (default no)

    Returns:
        The state object
    """
    return State(
        qpos=mjx_data.qpos,
        qvel=mjx_data.qvel,
        xpos=mjx_data.xpos,
        site_xpos=mjx_data.site_xpos,
        efc_force=jnp.where(return_contact_forces, mjx_data._impl.efc_force, jnp.zeros(())),
    )

class ConstraintViolations(PyTreeNode):
    """
    Track violations of the equality constraints, joint limit constraints, and contact constraints

    Attributes:
        equality_violation: The sum of the absolute value of the equality constraints  (1,)
        limit_violations: The sum of the absolute value of the joint limit constraints (1,)
        contact_violations: The sum of the absolute value of the contact constraints   (1,)
    """

    equality_violation: jax.Array
    limit_violations: jax.Array
    friction_violations: jax.Array
    contact_violations: jax.Array

# Params

In [17]:
# model_path = "/mnt/sdb2/DeepProprioception/MyoSuite/muscle_mimic/muscle_mimic.xml"
model_path = "/home/seojin/Tools/biomechanics/MonocularBiomechanics/monocular_demos/biomechanics_mjx/data/humanoid/humanoid_torque.xml"
scale_param_path = "/mnt/ext1/seojin/temp/scale/scaled_param.npz"
site_offset_path = "/mnt/ext1/seojin/temp/scale/site_offset_param.npz"

# Constants

In [18]:
scale_params = np.load(scale_param_path)
site_params = np.load(site_offset_path)

site_offsets = jnp.array(site_params["offset"])

# Initialize model

In [19]:
# Mujoco
model = mujoco.MjModel.from_xml_path(model_path)
body_names = [model.body(i).name for i in range(len(model.body_pos))]
site_names = [model.site(i).name for i in range(model.nsite)]
joint_names = [str(mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, joint_id)) for joint_id in range(model.njnt)]

# Jax
mjx_model = mjx.put_model(model)

# Joint
joint_angles = np.zeros(len(joint_names))

# Scale model as much as scale_params

In [20]:
scaled_model = mjx_model.replace(body_pos = scale_params["body_pos"])
scaled_model = scaled_model.replace(site_pos = scale_params["site_pos"])
scaled_model = scaled_model.replace(geom_pos = scale_params["geom_pos"])
scaled_model = scaled_model.replace(geom_size = scale_params["geom_size"])
scaled_model = scaled_model.replace(tendon_length0 = scale_params["tendon_length0"])

# Adjust site pose as much as site offset is given

In [21]:
scaled_model = mjx.put_model(model)
scaled_model = scaled_model.replace(site_pos = scaled_model.site_pos + site_offsets)

# Make state

In [9]:
mjx_data = mjx.make_data(scaled_model)

forward = functools.partial(mjx.forward, scaled_model)
qpos = mjx_data.qpos.at[:].set(joint_angles)
mjx_data = mjx_data.replace(qpos=qpos)
mjx_data = forward(mjx_data)

In [10]:
return_contact_forces = False
state = reduce_state(mjx_data, return_contact_forces)

# Constraint

In [11]:
ne = mjx_data._impl.ne    # number of equality constraints
nf = mjx_data._impl.nf    # number of friction constraints
nl = mjx_data._impl.nl    # number of limit constraints
nc = mjx_data._impl.ncon  # number of contact constraint counts

In [12]:
dx = mjx.make_constraint(scaled_model, mjx_data)

In [13]:
equality_violation = jax.lax.cond(ne > 0, lambda: jnp.mean(dx._impl.efc_aref[:ne] ** 2), lambda: jnp.array(0.0, dx._impl.efc_aref.dtype))
friction_violations = jax.lax.cond(nf > 0, lambda: jnp.mean(dx._impl.efc_aref[ne : ne + nf] ** 2), lambda: jnp.array(0.0, dx._impl.efc_aref.dtype))
limit_violations = jax.lax.cond(nl > 0, lambda: jnp.mean(dx._impl.efc_aref[ne + nf : ne + nf + nl] ** 2), lambda: jnp.array(0.0, dx._impl.efc_aref.dtype))

contact_violations = jax.lax.cond(
    nc > 0, lambda: jnp.mean(dx._impl.efc_aref[ne + nf + nl : ne + nf + nl + nc] ** 2), lambda: jnp.array(0.0, dx._impl.efc_aref.dtype)
)

In [14]:
constraints = ConstraintViolations(
    equality_violation=equality_violation,
    friction_violations=friction_violations,
    limit_violations=limit_violations,
    contact_violations=contact_violations,
)

In [15]:
constraints

ConstraintViolations(equality_violation=Array(0., dtype=float64), limit_violations=Array(0., dtype=float64), friction_violations=Array(0., dtype=float64), contact_violations=Array(344.61037147, dtype=float64))